
# Chapter 9: Some Group Theory

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 9, printed pp. 233-250, PDF pp. 251-268.

**Source inspection note.** The assigned source span was inspected with `pdftotext` for orientation only. In this local PDF build, physical `pdftotext` pages 249-266 contain printed pages 233-250, while the requested physical pages 251-268 begin at printed page 235 and continue into the opening of Chapter 10. The notebook follows the assigned printed span and its chapter structure: free products, free groups, presentations, and free abelian groups.

This chapter is the algebraic workbench for the next topological computations. The immediate topological motivation is that a loop in a union or a wedge often decomposes into pieces that live in different subspaces. Algebraically, those pieces should be multiplied without forcing them to commute. Later, when a relation comes from an overlap or an attaching map, we need a language that says exactly which words are identified. The four tools here do that in increasing order of control: free products combine existing groups without extra cross-relations; free groups create words from bare generators; presentations impose selected relators; free abelian groups record additive data after all commutators have been forgotten.



## Chapter Goal

By the end of this notebook, you should be able to translate the algebraic objects of Chapter 9 into computable normal forms and invariants. The central skill is to look at a word and ask which rules are allowed to shorten, rearrange, or identify it. In a free product, only adjacent letters from the same factor group may multiply, and identity letters disappear. In a free group, adjacent inverse letters cancel, but unrelated generators do not commute. In a presentation, a relator declares a chosen word to be invisible in the quotient, so equality becomes a question about reductions plus inserted relators. In a free abelian group, the word order is discarded and only the integer exponent vector remains.

## Computational Translation Guide

| Book concept | Computational model in this notebook | Invariant to inspect |
| --- | --- | --- |
| Word in a free product | Tagged finite tuple, with each tag naming its factor group | Adjacent equal tags reduce; different tags stay separated |
| Reduced representative | Deterministic stack normal form | Applying the reducer twice changes nothing |
| Free group on generators | Reduced words over letters and formal inverses | Cayley graph is a tree, so no hidden relation closes a loop |
| Universal property | Extension rule from generator images | The same generator map determines exactly one homomorphism |
| Presentation `<S | R>` | Free words modulo a chosen set of relator moves | Relators collapse selected loops, not arbitrary words |
| Free abelian group | Finitely supported integer vector on a basis | Addition of vectors matches concatenation after abelianization |
| Rank and rank-nullity | Integer matrix between free abelian groups | `rank(domain) = rank(image) + rank(kernel)` after torsion is ignored |

## Visual Storyboard

The visual path runs from words to spaces of words. We first reduce free-product words, then expand the free group into a tree where no relation is hidden, then fold selected relators into quotient cycles, and finally forget order by abelianizing to an integer lattice. The applied lab turns those pictures into rank and kernel checks for maps between free abelian groups.

## Library Routing

This is a chapter about algebraic structure, not metric surfaces. NetworkX is the right tool for normal-form and proof-dependency graphs because the learner needs to see adjacency, trees, and quotient cycles. Matplotlib gives durable static PNG diagrams for the finite pieces that should render in every environment. Plotly is useful for the larger free-group Cayley tree because the learner can pan and inspect labels in a standalone HTML artifact. SymPy handles exact integer matrix rank and kernel checks for the free abelian lab. Pandas records small tables of word images and routing decisions. The existing course helpers provide artifact paths, display hooks, and basic word/abelianization utilities.


In [ ]:

from __future__ import annotations

from collections import Counter, deque
from itertools import product
from pathlib import Path
import json
import math
import sys


def locate_book_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates += [candidate / "Introduction-to-Topological-Manifolds" for candidate in candidates]
    for candidate in candidates:
        if (candidate / "source_map.json").exists() and (candidate / "utils").exists():
            return candidate
    raise RuntimeError("Could not locate Introduction-to-Topological-Manifolds book root")


BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.topology import abelianization_vector, word_reduce  # noqa: E402
from utils.validation import image_stats, relative  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import networkx as nx  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import sympy as sp  # noqa: E402

UNIT_KEY = "chapter-09-some-group-theory"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"


def display_path(path: Path) -> Path:
    path = Path(path)
    try:
        return path.resolve().relative_to(Path.cwd().resolve())
    except ValueError:
        return path

artifact_paths: list[Path] = []

visual_storyboard = {
    "chapter": "Chapter 9 Some Group Theory",
    "source_span": {
        "printed_pages": "233-250",
        "assigned_pdf_pages": "251-268",
        "pdftotext_note": "Local physical pdftotext pages 249-266 contain the printed chapter span; requested pages 251-268 were also spot-checked.",
    },
    "items": [
        {
            "concept": "free product normal form",
            "representation": "Z/2 * Z/2 reduced-word Cayley prefixes",
            "library": "networkx + matplotlib",
            "artifact": "figures/free-product-z2-z2-cayley-prefixes.png",
            "inspection_target": "Only adjacent letters from the same factor cancel; beta gamma and gamma beta remain distinct.",
            "validation": "sampled associativity, idempotent reduction, and noncommuting products",
        },
        {
            "concept": "free group has no hidden relations",
            "representation": "ball in the Cayley tree of F(a,b)",
            "library": "networkx + plotly",
            "artifact": "html/free-group-cayley-tree.html",
            "inspection_target": "Each reduced word is reached by a unique path from the identity.",
            "validation": "node count, edge count, acyclicity, and inverse cancellation",
        },
        {
            "concept": "presentations impose relators",
            "representation": "infinite cyclic line folded to a finite cycle plus commutator check",
            "library": "matplotlib + course abelianization helper",
            "artifact": "figures/presentation-relators-as-quotients.png",
            "inspection_target": "A relator closes a selected loop; commutator relators force abelian behavior.",
            "validation": "r^5 normalizes to identity in C5 and the commutator has zero abelianized vector",
        },
        {
            "concept": "free abelian group as exponent lattice",
            "representation": "many free words mapped to one Z^2 vector",
            "library": "matplotlib + pandas",
            "artifact": "figures/free-abelian-abelianization-lattice.png",
            "inspection_target": "Order information disappears while integer exponent totals remain.",
            "validation": "abelianization is additive under word concatenation",
        },
        {
            "concept": "rank for finitely generated abelian groups",
            "representation": "integer matrix rank-nullity lab",
            "library": "sympy + matplotlib",
            "artifact": "figures/rank-nullity-free-abelian-lab.png",
            "inspection_target": "Kernel directions plus image rank account for the domain rank.",
            "validation": "exact SymPy ranks and primitive integer kernel basis",
        },
        {
            "concept": "proof dependencies",
            "representation": "directed dependency graph for Chapter 9 tools",
            "library": "networkx + matplotlib",
            "artifact": "figures/chapter-09-proof-dependency-map.png",
            "inspection_target": "Normal forms support universal properties, which support presentations and rank tools.",
            "validation": "directed acyclic graph with named source and sink nodes",
        },
    ],
}

storyboard_path = save_json(visual_storyboard, CHECKS / "visual-storyboard.json")
artifact_paths.append(storyboard_path)

routing_rows = [
    {
        "concept": item["concept"],
        "representation": item["representation"],
        "library": item["library"],
        "why": item["inspection_target"],
        "fallback": "static markdown table" if "plotly" in item["library"] else "plain table of normal forms",
    }
    for item in visual_storyboard["items"]
]
routing_path = save_csv(routing_rows, TABLES / "chapter-09-library-routing.csv")
artifact_paths.append(routing_path)

pd.DataFrame(routing_rows)


In [ ]:
display_artifact(display_path(storyboard_path), width=760, height=160)



## Free Products: Normal Forms With Tags

A direct product makes different factors commute because multiplication happens coordinate by coordinate. A free product does the opposite: it places elements from different factor groups in one word but keeps their factor tags visible. The only local moves are the ones already justified inside a single factor group. If two neighboring letters came from the same factor, multiply them there. If an identity appears, delete it. Otherwise the boundary between different factor groups is protected.

The practical theorem behind this construction is the normal-form statement: every equivalence class has one reduced representative. Computationally, this is a stack algorithm. Read letters from left to right. If the new letter is an identity, skip it. If it belongs to the same factor as the last surviving letter, replace the last letter by the product in that factor, deleting the result if it is the identity. If the factor tag differs, append the new letter. The output is reduced, and the proposition says that this output depends only on the equivalence class, not on accidental choices made during reduction.

The smallest useful model is `Z/2 * Z/2`. Each factor has one nonidentity element, and that element is its own inverse. The reduced words are alternating strings. This already shows why the construction is nonabelian: `beta gamma` and `gamma beta` are different reduced words. The next cell builds the finite prefix of its Cayley graph and records the invariants that make the picture a proof scaffold rather than a decoration.


In [ ]:

def fp_label(word: tuple[tuple[str, int], ...]) -> str:
    if not word:
        return "1"
    names = {"B": "beta", "G": "gamma"}
    return " ".join(names[group] for group, exponent in word)


def z2_reduce(raw_word: list[tuple[str, int]] | tuple[tuple[str, int], ...]):
    # Return the reduced word and a trace for a word in Z/2 * Z/2.
    stack: list[tuple[str, int]] = []
    trace: list[dict[str, str]] = []
    for group, exponent in raw_word:
        exponent %= 2
        before = tuple(stack)
        if exponent == 0:
            action = f"delete identity from {group}"
        elif stack and stack[-1][0] == group:
            stack.pop()
            action = f"{group} followed by {group} multiplies to identity"
        else:
            stack.append((group, 1))
            action = f"append {group}"
        trace.append({"before": fp_label(before), "letter": group, "action": action, "after": fp_label(tuple(stack))})
    return tuple(stack), trace


def z2_multiply(left, right):
    return z2_reduce(list(left) + list(right))[0]


B = (("B", 1),)
G = (("G", 1),)
noncommuting_pair = {"beta_gamma": fp_label(z2_multiply(B, G)), "gamma_beta": fp_label(z2_multiply(G, B))}

raw_word = [("B", 1), ("B", 1), ("G", 1), ("B", 1), ("G", 1), ("G", 1), ("B", 1)]
reduced_word, reduction_trace = z2_reduce(raw_word)

max_depth = 7
nodes = {()}
frontier = [()]
for _ in range(max_depth):
    new_frontier = []
    for node in frontier:
        for gen in [B, G]:
            target = z2_multiply(node, gen)
            if len(target) <= max_depth and target not in nodes:
                nodes.add(target)
                new_frontier.append(target)
    frontier = new_frontier

G_prefix = nx.Graph()
for node in nodes:
    G_prefix.add_node(node, label=fp_label(node), length=len(node))
for node in nodes:
    for gen in [B, G]:
        target = z2_multiply(node, gen)
        if target in nodes:
            G_prefix.add_edge(node, target, generator=fp_label(gen))

positions = {(): (0, 0)}
for node in nodes:
    if not node:
        continue
    sign = 1 if node[0][0] == "B" else -1
    positions[node] = (sign * len(node), 0.18 * ((-1) ** len(node)))

fig, ax = plt.subplots(figsize=(10, 3.8))
node_colors = []
for node in G_prefix.nodes:
    if not node:
        node_colors.append("#f2f2f2")
    elif node[-1][0] == "B":
        node_colors.append("#4c78a8")
    else:
        node_colors.append("#f58518")
nx.draw_networkx_edges(G_prefix, positions, ax=ax, width=1.6, edge_color="#555555")
nx.draw_networkx_nodes(G_prefix, positions, ax=ax, node_color=node_colors, node_size=520, edgecolors="#222222")
labels = {node: ("1" if not node else "b" if node[-1][0] == "B" else "g") for node in G_prefix.nodes}
nx.draw_networkx_labels(G_prefix, positions, labels=labels, ax=ax, font_size=9, font_color="#111111")
ax.set_title("Finite prefix of the Cayley graph for Z/2 * Z/2")
ax.text(0, -0.56, "Alternating rays encode unique reduced words; beta gamma and gamma beta sit on opposite sides.", ha="center")
ax.set_axis_off()
free_product_png = save_matplotlib(fig, FIGURES / "free-product-z2-z2-cayley-prefixes.png")
plt.close(fig)
artifact_paths.append(free_product_png)

words_for_sampling = sorted(nodes, key=lambda word: (len(word), fp_label(word)))
for a, b, c in product(words_for_sampling[:9], repeat=3):
    assert z2_multiply(z2_multiply(a, b), c) == z2_multiply(a, z2_multiply(b, c))
assert z2_reduce(reduced_word)[0] == reduced_word
assert z2_multiply(B, G) != z2_multiply(G, B)

free_product_checks = {
    "raw_word": [group for group, exponent in raw_word],
    "reduced_word": fp_label(reduced_word),
    "trace": reduction_trace,
    "noncommuting_pair": noncommuting_pair,
    "graph_nodes": G_prefix.number_of_nodes(),
    "graph_edges": G_prefix.number_of_edges(),
    "associativity_samples": 9**3,
    "reduction_idempotent": True,
}
free_product_check_path = save_json(free_product_checks, CHECKS / "free-product-normal-form-checks.json")
artifact_paths.append(free_product_check_path)

display_artifact(display_path(free_product_png), width=860)
pd.DataFrame(reduction_trace)



## Free Groups: A Tree of Reduced Words

A free group starts with a set `S` and does not assume any multiplication table among distinct generators. The only built-in relation is that every letter cancels with its formal inverse. For a singleton, this recovers the infinite cyclic group: the reduced words are integer powers of one generator. For two generators, the picture changes from a line to a branching tree. From the identity there are four first moves, `a`, `a^-1`, `b`, and `b^-1`. After a move, the one edge immediately back is forbidden if the word is to stay reduced, so every non-root vertex has three outward choices.

This tree is the computational signature of freeness. A loop in the Cayley graph would encode a nontrivial reduced word equal to the identity, which would be a hidden relation. The finite ball below is not merely a pretty graph: its node count and acyclicity are sanity checks for the normal-form theorem. The universal property then says that to define a homomorphism out of `F(S)`, it is enough to choose images for the letters of `S`; the reduction rules force the images of inverses and products.


In [ ]:

import plotly.graph_objects as go

FREE_GENERATORS = ("a", "a^-1", "b", "b^-1")


def reduced_append(word: tuple[str, ...], letter: str) -> tuple[str, ...]:
    return tuple(word_reduce([*word, letter]))


def word_text(word: tuple[str, ...]) -> str:
    return "1" if not word else " ".join(word)


def free_group_ball(depth: int = 4) -> nx.Graph:
    graph = nx.Graph()
    root: tuple[str, ...] = ()
    graph.add_node(root, depth=0, label="1")
    queue = deque([root])
    while queue:
        word = queue.popleft()
        if len(word) >= depth:
            continue
        for letter in FREE_GENERATORS:
            new_word = reduced_append(word, letter)
            if len(new_word) == len(word) + 1:
                if new_word not in graph:
                    graph.add_node(new_word, depth=len(new_word), label=word_text(new_word))
                    queue.append(new_word)
                graph.add_edge(word, new_word, label=letter)
    return graph


free_tree = free_group_ball(depth=4)
free_tree_pos = nx.spring_layout(free_tree, seed=23, k=0.95, iterations=250)

edge_x, edge_y = [], []
for source, target in free_tree.edges:
    x0, y0 = free_tree_pos[source]
    x1, y1 = free_tree_pos[target]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

node_x = [free_tree_pos[node][0] for node in free_tree.nodes]
node_y = [free_tree_pos[node][1] for node in free_tree.nodes]
node_depth = [free_tree.nodes[node]["depth"] for node in free_tree.nodes]
hover = [free_tree.nodes[node]["label"] for node in free_tree.nodes]
short_text = ["1" if not node else "" for node in free_tree.nodes]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1, color="#888"), hoverinfo="skip"))
fig.add_trace(
    go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=short_text,
        textposition="middle center",
        marker=dict(size=8, color=node_depth, colorscale="Viridis", showscale=True, colorbar=dict(title="length")),
        hovertext=hover,
        hoverinfo="text",
    )
)
fig.update_layout(
    title="Ball of radius 4 in the Cayley tree of F(a,b)",
    width=860,
    height=620,
    showlegend=False,
    margin=dict(l=20, r=20, t=50, b=20),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)
free_group_html = save_plotly_html(fig, HTML / "free-group-cayley-tree.html")
artifact_paths.append(free_group_html)

expected_nodes = 1 + 4 * sum(3**k for k in range(4))
assert free_tree.number_of_nodes() == expected_nodes
assert free_tree.number_of_edges() == free_tree.number_of_nodes() - 1
assert nx.is_tree(free_tree)
assert reduced_append(("a", "b"), "b^-1") == ("a",)
assert tuple(word_reduce(["a", "b", "b^-1", "a^-1"])) == ()

free_group_checks = {
    "depth": 4,
    "node_count": free_tree.number_of_nodes(),
    "expected_node_count": expected_nodes,
    "edge_count": free_tree.number_of_edges(),
    "is_tree": nx.is_tree(free_tree),
    "sample_cancellation": "a b b^-1 a^-1 -> 1",
    "extension_rule_example": {
        "generator_images": {"a": "h", "b": "k"},
        "word": "a b^-1 a",
        "forced_image": "h k^-1 h",
    },
}
free_group_check_path = save_json(free_group_checks, CHECKS / "free-group-cayley-tree-checks.json")
artifact_paths.append(free_group_check_path)

display_artifact(display_path(free_group_html), width=880, height=660)
free_group_checks



## Presentations: Free First, Then Impose Relators

A presentation separates two jobs. First, take a free group on named generators, so every word has an unambiguous reduced form. Second, choose a set of relators and declare each relator to represent the identity. The resulting group is a quotient of the free group. This is exactly the style needed for surface groups in the next chapter: generators come from loops, while relators come from the way boundary paths or overlaps are identified.

A presentation should be read as a machine for creating equality, not as a complete multiplication table. The presentation `<r | r^5>` folds the infinite cyclic line into a five-cycle. The presentation `<a,b | a b a^-1 b^-1>` forces the commutator to vanish; once that happens, `a b` and `b a` have the same image. By contrast, in the free group `F(a,b)` the reduced words `a b` and `b a` remain different. The point is not that presentations make all computations easy. The chapter explicitly warns that the word problem and isomorphism problem for finite presentations are deep limitations. For the controlled presentations used in this course, however, exponent vectors, relator checks, and quotient diagrams give reliable local tests.


In [ ]:

def cyclic_normal_form(exponent: int, modulus: int) -> int:
    return exponent % modulus


modulus = 5
line_nodes = list(range(-5, 6))
cycle_nodes = list(range(modulus))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.plot(line_nodes, np.zeros(len(line_nodes)), color="#4c78a8", linewidth=2)
ax.scatter(line_nodes, np.zeros(len(line_nodes)), s=80, color="#4c78a8", edgecolor="#222222", zorder=3)
for x in line_nodes:
    ax.text(x, 0.12, f"r^{x}", ha="center", fontsize=8)
ax.annotate("multiply by r", xy=(2.8, -0.18), xytext=(0.2, -0.18), arrowprops=dict(arrowstyle="->"), ha="center")
ax.set_title("Free group on one generator: infinite line")
ax.set_ylim(-0.45, 0.45)
ax.set_axis_off()

ax = axes[1]
angles = np.linspace(0, 2 * np.pi, modulus, endpoint=False) + np.pi / 2
cycle_positions = np.column_stack([np.cos(angles), np.sin(angles)])
for i in range(modulus):
    j = (i + 1) % modulus
    ax.annotate("", xy=cycle_positions[j], xytext=cycle_positions[i], arrowprops=dict(arrowstyle="->", color="#666666", lw=1.5))
ax.scatter(cycle_positions[:, 0], cycle_positions[:, 1], s=220, color="#54a24b", edgecolor="#222222", zorder=3)
for i, (x, y) in enumerate(cycle_positions):
    ax.text(x, y, f"r^{i}", ha="center", va="center", fontsize=9)
ax.text(0, -1.35, "Relator r^5 = 1 closes the path after five steps.", ha="center")
ax.set_title("Presentation <r | r^5>: quotient cycle")
ax.set_aspect("equal")
ax.set_axis_off()

presentation_png = save_matplotlib(fig, FIGURES / "presentation-relators-as-quotients.png")
plt.close(fig)
artifact_paths.append(presentation_png)

commutator_word = ["a", "b", "a^-1", "b^-1"]
commutator_vector = abelianization_vector(commutator_word, ["a", "b"])
assert cyclic_normal_form(5, modulus) == 0
assert cyclic_normal_form(17, modulus) == 2
assert commutator_vector == {"a": 0, "b": 0}
assert tuple(word_reduce(["a", "b"])) != tuple(word_reduce(["b", "a"]))

presentation_rows = [
    {"presentation": "<r | r^5>", "word": "r^17", "normal_data": "r^2", "check": "17 mod 5 = 2"},
    {"presentation": "<a,b | a b a^-1 b^-1>", "word": "a b a^-1 b^-1", "normal_data": str(commutator_vector), "check": "commutator maps to zero vector"},
    {"presentation": "F(a,b)", "word": "a b versus b a", "normal_data": "different reduced words", "check": "no commutator relator present"},
]
presentation_table_path = save_csv(presentation_rows, TABLES / "presentation-normal-form-examples.csv")
artifact_paths.append(presentation_table_path)

presentation_checks = {
    "cyclic_modulus": modulus,
    "r_5_is_identity": cyclic_normal_form(5, modulus) == 0,
    "r_17_normal_form": cyclic_normal_form(17, modulus),
    "commutator_abelianization": commutator_vector,
    "free_group_ab_vs_ba_equal": tuple(word_reduce(["a", "b"])) == tuple(word_reduce(["b", "a"])),
}
presentation_check_path = save_json(presentation_checks, CHECKS / "presentation-relator-checks.json")
artifact_paths.append(presentation_check_path)

display_artifact(display_path(presentation_png), width=900)
pd.DataFrame(presentation_rows)



## Free Abelian Groups: Words Become Integer Vectors

Free abelian groups are the additive analogue of free groups, but there is one essential difference: order no longer matters. An element of the free abelian group on a set `S` is a finite integer linear combination of elements of `S`. For a finite basis, this is just an integer lattice. The characteristic property says that any set map from the basis into an abelian group extends uniquely to a homomorphism. In computational terms, once the basis images are chosen, the image of a vector is the same integer linear combination of those images.

The bridge from free groups to free abelian groups is abelianization. A reduced word in `F(a,b)` maps to the vector recording the total exponent of `a` and the total exponent of `b`. This map forgets order and all commutators. That is a feature when the invariant being studied is homological: Chapter 13 will use chains and homology groups where rank and torsion matter, while the exact order in a nonabelian loop product is no longer visible. The plot below sends all reduced words of length at most five in `F(a,b)` to `Z^2`. Multiple free words can land on the same lattice point; that pile-up is the loss of nonabelian information.


In [ ]:

def generate_free_words(max_len: int, alphabet: tuple[str, ...] = FREE_GENERATORS) -> list[tuple[str, ...]]:
    words = [()]
    queue = deque([()])
    while queue:
        word = queue.popleft()
        if len(word) >= max_len:
            continue
        for letter in alphabet:
            new_word = reduced_append(word, letter)
            if len(new_word) == len(word) + 1:
                words.append(new_word)
                queue.append(new_word)
    return words


free_words = generate_free_words(5)
word_vectors = []
for word in free_words:
    vector = abelianization_vector(word, ["a", "b"])
    word_vectors.append((word, (vector["a"], vector["b"])))

vector_counts = Counter(vector for word, vector in word_vectors)
xs = np.array([vector[0] for vector in vector_counts])
ys = np.array([vector[1] for vector in vector_counts])
sizes = np.array([80 + 28 * count for count in vector_counts.values()])
colors = np.array(list(vector_counts.values()))

fig, ax = plt.subplots(figsize=(7.2, 7.2))
for x in range(xs.min() - 1, xs.max() + 2):
    ax.axvline(x, color="#dddddd", linewidth=0.7, zorder=0)
for y in range(ys.min() - 1, ys.max() + 2):
    ax.axhline(y, color="#dddddd", linewidth=0.7, zorder=0)
scatter = ax.scatter(xs, ys, s=sizes, c=colors, cmap="magma", edgecolor="#222222", alpha=0.88)
for (x, y), count in vector_counts.items():
    if count >= 4:
        ax.text(x, y, str(count), ha="center", va="center", color="white", fontsize=8, weight="bold")
ax.set_title("Abelianization sends reduced words in F(a,b) to exponent vectors in Z^2")
ax.set_xlabel("total exponent of a")
ax.set_ylabel("total exponent of b")
ax.set_aspect("equal", adjustable="box")
fig.colorbar(scatter, ax=ax, shrink=0.75, label="number of words landing here")
free_abelian_png = save_matplotlib(fig, FIGURES / "free-abelian-abelianization-lattice.png")
plt.close(fig)
artifact_paths.append(free_abelian_png)

sample_words = [
    ("a b", ("a", "b")),
    ("b a", ("b", "a")),
    ("a b a^-1", ("a", "b", "a^-1")),
    ("b^-1 a b", ("b^-1", "a", "b")),
    ("a b b^-1 a^-1", ("a", "b", "b^-1", "a^-1")),
]
sample_rows = []
for label, word in sample_words:
    reduced = tuple(word_reduce(word))
    vector = abelianization_vector(reduced, ["a", "b"])
    sample_rows.append({"free_word": label, "reduced_word": word_text(reduced), "a_total": vector["a"], "b_total": vector["b"]})
word_image_table_path = save_csv(sample_rows, TABLES / "free-abelian-word-images.csv")
artifact_paths.append(word_image_table_path)

u = ("a", "b", "a^-1")
v = ("b^-1", "a")
uv_reduced = tuple(word_reduce([*u, *v]))
vec_u = abelianization_vector(u, ["a", "b"])
vec_v = abelianization_vector(v, ["a", "b"])
vec_uv = abelianization_vector(uv_reduced, ["a", "b"])
assert vec_uv == {generator: vec_u[generator] + vec_v[generator] for generator in ["a", "b"]}
assert abelianization_vector(["a", "b"], ["a", "b"]) == abelianization_vector(["b", "a"], ["a", "b"])
assert tuple(word_reduce(["a", "b"])) != tuple(word_reduce(["b", "a"]))

free_abelian_checks = {
    "words_sampled": len(free_words),
    "lattice_points_hit": len(vector_counts),
    "max_words_on_one_vector": max(vector_counts.values()),
    "ab_and_ba_same_vector": True,
    "ab_and_ba_same_free_word": False,
    "additivity_check": {"u": vec_u, "v": vec_v, "uv": vec_uv},
}
free_abelian_check_path = save_json(free_abelian_checks, CHECKS / "free-abelian-abelianization-checks.json")
artifact_paths.append(free_abelian_check_path)

display_artifact(display_path(free_abelian_png), width=760)
pd.DataFrame(sample_rows)



## Proof and Invariant Scaffolds

The proofs in this chapter repeatedly use the same pattern. Construct a large formal object, define a normal form or invariant, and then prove that the construction has the desired universal property. For free products, the normal form is the unique reduced word. For free groups, it is the reduced word in generators and inverses. For presentations, the quotient map from the free group makes relators trivial. For free abelian groups, the normal form is a finite-support integer vector, and rank is the invariant that survives a change of basis.

This dependency graph is a compact reading map for the chapter. It is directed because later claims depend on earlier normal forms. It is acyclic because the logical flow does not use presentations to prove free-product normal forms or rank-nullity to define free groups. The graph is also a useful self-check when using the chapter in topology: if a van Kampen computation names generators and relators, it relies on the free group and presentation boxes; if a homology computation uses ranks, it relies on the free abelian and rank boxes.


In [ ]:

dep_edges = [
    ("tagged words", "elementary reductions"),
    ("elementary reductions", "unique reduced word"),
    ("unique reduced word", "free product group law"),
    ("free product group law", "free product universal property"),
    ("free product universal property", "free group F(S)"),
    ("free group F(S)", "presentation <S | R>"),
    ("presentation <S | R>", "surface group computations"),
    ("finite-support vectors", "free abelian group ZS"),
    ("free abelian group ZS", "basis and rank"),
    ("basis and rank", "subgroups of finite-rank free abelian groups"),
    ("subgroups of finite-rank free abelian groups", "finitely generated torsion-free groups"),
    ("finitely generated torsion-free groups", "rank-nullity for abelian groups"),
    ("rank-nullity for abelian groups", "homology rank computations"),
    ("presentation <S | R>", "abelianization checks"),
    ("free abelian group ZS", "abelianization checks"),
]
D = nx.DiGraph()
D.add_edges_from(dep_edges)
assert nx.is_directed_acyclic_graph(D)
levels = {
    "tagged words": 0,
    "finite-support vectors": 0,
    "elementary reductions": 1,
    "unique reduced word": 2,
    "free product group law": 3,
    "free product universal property": 4,
    "free group F(S)": 5,
    "presentation <S | R>": 6,
    "surface group computations": 7,
    "abelianization checks": 7,
    "free abelian group ZS": 2,
    "basis and rank": 3,
    "subgroups of finite-rank free abelian groups": 4,
    "finitely generated torsion-free groups": 5,
    "rank-nullity for abelian groups": 6,
    "homology rank computations": 7,
}
pos = {}
level_counts = Counter(levels.values())
level_seen = Counter()
for node in nx.topological_sort(D):
    level = levels[node]
    index = level_seen[level]
    level_seen[level] += 1
    count = level_counts[level]
    y = 0 if count == 1 else (index - (count - 1) / 2)
    pos[node] = (level, -y)

fig, ax = plt.subplots(figsize=(14, 6.5))
nx.draw_networkx_edges(D, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, width=1.2, edge_color="#555555")
node_colors = ["#4c78a8" if "free" in node else "#f58518" if "rank" in node or "abelian" in node else "#72b7b2" for node in D.nodes]
nx.draw_networkx_nodes(D, pos, node_size=1650, node_color=node_colors, edgecolors="#222222", ax=ax)
nx.draw_networkx_labels(D, pos, font_size=8, ax=ax)
ax.set_title("Chapter 9 proof dependency map")
ax.set_axis_off()
dependency_png = save_matplotlib(fig, FIGURES / "chapter-09-proof-dependency-map.png")
plt.close(fig)
artifact_paths.append(dependency_png)

dependency_checks = {
    "nodes": D.number_of_nodes(),
    "edges": D.number_of_edges(),
    "is_dag": nx.is_directed_acyclic_graph(D),
    "source_nodes": [node for node, degree in D.in_degree() if degree == 0],
    "sink_nodes": [node for node, degree in D.out_degree() if degree == 0],
}
dependency_check_path = save_json(dependency_checks, CHECKS / "chapter-09-proof-dependency-map.json")
artifact_paths.append(dependency_check_path)

display_artifact(display_path(dependency_png), width=980)
dependency_checks



## Applied Lab: Rank-Nullity Over Free Abelian Groups

Free abelian groups of finite rank behave like integer lattices. A homomorphism `Z^n -> Z^m` is represented by an integer matrix. The image rank is the rank of the matrix over the rationals; the kernel rank is the number of independent integer directions killed by the matrix. The chapter's rank-nullity statement for finitely generated abelian groups says that, after torsion is accounted for, the familiar linear algebra formula still measures the free part.

The lab below studies a concrete homomorphism `f: Z^3 -> Z^2`. It is not trying to classify the quotient or compute Smith normal form; that would be a stronger invariant. Instead, it focuses on the rank information needed later in homology. The matrix has a one-dimensional kernel generated by a primitive integer vector. The two rows are independent, so the image rank is two. The final equality `3 = 2 + 1` is the free-abelian rank-nullity check.


In [ ]:

def primitive_integer_vector(vector: sp.Matrix) -> list[int]:
    entries = list(vector)
    denominators = [sp.denom(entry) for entry in entries]
    scale = int(sp.ilcm(*[int(denominator) for denominator in denominators])) if denominators else 1
    ints = [int(entry * scale) for entry in entries]
    gcd = abs(math.gcd(*ints)) if any(ints) else 1
    return [value // gcd for value in ints]


A = sp.Matrix([[2, 4, 6], [1, 1, 1]])
domain_rank = A.shape[1]
image_rank = int(A.rank())
nullspace = A.nullspace()
kernel_basis = [primitive_integer_vector(vector) for vector in nullspace]
kernel_rank = len(kernel_basis)
assert domain_rank == image_rank + kernel_rank
assert A * sp.Matrix(kernel_basis[0]) == sp.zeros(A.shape[0], 1)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
axes[0].imshow(np.array(A.tolist(), dtype=float), cmap="cividis")
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        axes[0].text(j, i, str(A[i, j]), ha="center", va="center", color="white", fontsize=12, weight="bold")
axes[0].set_xticks(range(A.shape[1]), labels=["e1", "e2", "e3"])
axes[0].set_yticks(range(A.shape[0]), labels=["row 1", "row 2"])
axes[0].set_title("Integer matrix for f: Z^3 -> Z^2")

kernel_vector = np.array(kernel_basis[0], dtype=float)
axes[1].axhline(0, color="#dddddd", linewidth=1)
axes[1].bar(["e1", "e2", "e3"], kernel_vector, color=["#4c78a8", "#f58518", "#54a24b"], edgecolor="#222222")
axes[1].set_title("Primitive kernel generator")
axes[1].set_ylabel("coefficient")
for index, value in enumerate(kernel_vector):
    axes[1].text(index, value + (0.12 if value >= 0 else -0.22), str(int(value)), ha="center")
fig.suptitle("Rank-nullity check: rank Z^3 = rank image + rank kernel = 2 + 1")
rank_png = save_matplotlib(fig, FIGURES / "rank-nullity-free-abelian-lab.png")
plt.close(fig)
artifact_paths.append(rank_png)

rank_rows = [
    {"quantity": "domain rank", "value": domain_rank, "method": "number of basis vectors in Z^3"},
    {"quantity": "image rank", "value": image_rank, "method": "exact SymPy matrix rank"},
    {"quantity": "kernel rank", "value": kernel_rank, "method": "dimension of exact nullspace"},
    {"quantity": "kernel basis", "value": str(kernel_basis), "method": "primitive integer nullspace basis"},
]
rank_table_path = save_csv(rank_rows, TABLES / "rank-nullity-free-abelian-lab.csv")
artifact_paths.append(rank_table_path)

rank_checks = {
    "matrix": [[int(A[i, j]) for j in range(A.shape[1])] for i in range(A.shape[0])],
    "domain_rank": domain_rank,
    "image_rank": image_rank,
    "kernel_rank": kernel_rank,
    "kernel_basis": kernel_basis,
    "rank_nullity_holds": domain_rank == image_rank + kernel_rank,
    "kernel_residual": [[int(value) for value in row] for row in (A * sp.Matrix(kernel_basis[0])).tolist()],
}
rank_check_path = save_json(rank_checks, CHECKS / "rank-nullity-free-abelian-lab.json")
artifact_paths.append(rank_check_path)

display_artifact(display_path(rank_png), width=860)
pd.DataFrame(rank_rows)



## Takeaways

Free products preserve the internal multiplication of each factor group while refusing to add commutation rules between different factors. The useful computational object is the reduced word, and uniqueness of reduced words is what makes the construction a group with a universal property.

Free groups are the special case where the factors are infinite cyclic groups generated by the chosen set. Their Cayley graphs are trees because the only reductions are inverse cancellations. This is why a map from the generating set into any group extends in exactly one way to a homomorphism from the free group.

Presentations start with a free group and then quotient by relators. A relator should be treated as a specified loop that collapses, not as permission to rearrange every word unless the relator implies such a rearrangement. The examples `<r | r^n>` and `<a,b | a b a^-1 b^-1>` show two common outcomes: cyclic wrapping and commutativity.

Free abelian groups replace reduced noncommutative words with integer coefficient vectors. The basis gives unique coordinates, rank is well-defined for finite bases, and homomorphisms between finite-rank free abelian groups can be checked by exact integer matrices. This additive theory is the algebraic language that will reappear in homology.


In [ ]:

# Final sanity checks for Chapter 9 artifacts and invariants.
assert_artifacts(artifact_paths, min_bytes=80)

png_paths = [path for path in artifact_paths if path.suffix.lower() == ".png"]
png_stats = [image_stats(path) for path in png_paths]
assert png_stats, "Expected at least one PNG artifact"
assert all(stat["bytes"] > 1000 for stat in png_stats)
assert all(stat["max_channel_stddev"] > 2.0 for stat in png_stats)

json_paths = [path for path in artifact_paths if path.suffix.lower() == ".json"]
assert json_paths, "Expected at least one JSON check artifact"
for path in json_paths:
    json.loads(path.read_text(encoding="utf-8"))

core_checks = {
    "free_product_noncommuting": z2_multiply(B, G) != z2_multiply(G, B),
    "free_group_tree": nx.is_tree(free_tree),
    "presentation_commutator_zero_vector": commutator_vector == {"a": 0, "b": 0},
    "free_abelian_additivity": vec_uv == {generator: vec_u[generator] + vec_v[generator] for generator in ["a", "b"]},
    "rank_nullity": domain_rank == image_rank + kernel_rank,
    "dependency_graph_is_dag": nx.is_directed_acyclic_graph(D),
}
assert all(core_checks.values())

final_sanity = {
    "notebook": "chapter-09-some-group-theory/09-some-group-theory.ipynb",
    "source_span": "printed pp. 233-250; PDF pp. 251-268",
    "artifact_count_before_final": len(artifact_paths),
    "png_stats": png_stats,
    "json_checks": [relative(path, BOOK_ROOT) for path in json_paths],
    "core_checks": core_checks,
}
final_sanity_path = save_json(final_sanity, CHECKS / "final_sanity.json")
assert_artifacts([final_sanity_path], min_bytes=80)
artifact_paths.append(final_sanity_path)

display_artifact(display_path(final_sanity_path), width=760, height=160)
final_sanity
